In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.SILVER

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/silver"))

In [0]:
from pyspark.sql import SparkSession

produto="/Volumes/workspace/default/silver/products.csv"
spark.read.csv(
    produto,
    sep=";",
    header=True,
    inferSchema=True
  ).write.mode("overwrite").saveAsTable("silver_products")

In [0]:
%sql
select * from silver_products

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
#agrupa e ordena SQL
agrupa=spark.sql("""
                 select productname,
                 sum(UnitPrice)
                 from silver_products
                 group by productname
                 order by 2 desc
                 LIMIT 5
                 """)

#display(agrupa)      
# DF
df=spark.read.csv(produto,
                  sep=";",
                  header=True,
                  inferSchema=True
                  )
df_agrupado=df.groupBy("productname").agg(sum("UnitPrice").alias("Total_price"))
df_agrupado=df_agrupado.orderBy(col("Total_price").desc()).limit(5)       
display(df_agrupado)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/silver"))

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *
from pyspark.sql.types import DoubleType
vendas="/Volumes/workspace/default/silver/orders.csv"
detalhes="/Volumes/workspace/default/silver/orderdetails.csv"
prod="/Volumes/workspace/default/silver/products.csv"

df_venda=spark.read.csv(
    vendas,
    sep=";",
    header=True,
    inferSchema=True
  )
df_det=spark.read.csv(
    detalhes,
    sep=";",
    header=True,
    inferSchema=True
  )
df_prod=spark.read.csv(
    prod,
    sep=";",
    header=True,
    inferSchema=True
  )

#df_total = df_det.join(df_prod, on="ProductID", how="inner")
df_prod_renamed = df_prod.withColumnRenamed("UnitPrice", "ProductUnitPrice")
df_venda_renamed = df_det.withColumnRenamed("UnitPrice", "OrderUnitPrice")
df_vendido_rename = df_venda.withColumnRenamed("UnitPrice", "Preco_unitario")

# Fazer o join usando a coluna comum (CustomerID ou ProductID)
df_total = df_venda_renamed.join(df_prod_renamed, on="ProductID", how="inner")

# Agrupar e somar os preços unitários (agora sem ambiguidade)
# A coluna `ProductUnitPrice` já foi renomeada para ser única
# Além disso, garantir que o tipo de dado seja numérico (DoubleType)
df_aggr = df_total.groupBy("ProductName").agg(
    sum(col("ProductUnitPrice").cast(DoubleType())).alias("Total_price")
).orderBy(col("Total_price").desc())

# Exibir o resultado
display(df_aggr)
display(df_venda)


In [0]:
display(df_venda.limit(1))
display(df_prod.limit(1))
display(df_det.limit(1))

In [0]:
# Remover recursivamente o diretório existente
dbutils.fs.rm("/Volumes/workspace/default/silver", recurse=True)

# Gravar a tabela Delta em seguida
 spark.sql("""
  CREATE OR REPLACE TEMPORARY VIEW products_view
  USING CSV
  OPTIONS (
    path "/Volumes/workspace/default/silver/products.csv",
    header "true",
    inferSchema "true",
    delimiter ";"  -- Especifica que o separador é o ponto e vírgula
  )
""")


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import *


detalis="/Volumes/workspace/default/silver/orderdetails.csv"
prodct="/Volumes/workspace/default/silver/products.csv"
df_det=spark.read.csv(
    detalis,
    sep=";",
    header=True,
    inferSchema=True
  )
df_prod=spark.read.csv(
    prodct,
    sep=";",
    header=True,
    inferSchema=True
  )
#df_det = df_det.dropna(subset=["productname","UnitPrice","OrderID"])
#df_prod = df_prod.dropna(subset=["productname","UnitPrice","OrderID"])


df_aggr = spark.sql("SELECT * FROM products_view LIMIT 4")
#salvando em parquet
df_prod.write.mode("overwrite").partitionBy("CategoryID").delta("/Volumes/workspace/default/silver")
#salvando em delta
df_prod.write.format("delta").mode("overwrite").partitionBy("productname").save("/Volumes/workspace/default/silver")

display(df_aggr)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, avg, countDistinct, year, month

# Inicializa sessão Spark
spark = SparkSession.builder.appName("TesteActDigital").getOrCreate()

# === 1. Ingestão DataFrame clientes e transacoes ===
clientes = spark.read.csv("/mnt/data/clientes.csv", header=True, inferSchema=True) 
transacoes = spark.read.csv("/mnt/data/transacoes.csv", header=True, inferSchema=True)

# Validação simples
transacoes = transacoes.dropna(subset=["id_cliente", "data", "valor"])

# === 2. Transformação ===
# Extrair ano e mês
transacoes = transacoes.withColumn("ano", year(col("data"))) \
                       .withColumn("mes", month(col("data")))

# Total gasto por cliente/mês
gasto_mensal = transacoes.groupBy("id_cliente", "ano", "mes") \
    .agg(_sum("valor").alias("gasto_total"))

# Top 10 clientes acumulado
top_clientes = transacoes.groupBy("id_cliente") \
    .agg(_sum("valor").alias("gasto_acumulado")) \
    .orderBy(col("gasto_acumulado").desc()) \
    .limit(10)

# KPIs
kpis = transacoes.groupBy("ano", "mes").agg(
    avg("valor").alias("gasto_medio_cliente"),
    _sum("valor").alias("gasto_total_mes"),
    countDistinct("id_cliente").alias("clientes_ativos")
)

# === 3. Saída ===
gasto_mensal.write.mode("overwrite").partitionBy("ano", "mes").parquet("/mnt/silver/gasto_mensal")
top_clientes.write.mode("overwrite").parquet("/mnt/gold/top_clientes")
kpis.write.mode("overwrite").parquet("/mnt/gold/kpis")

# === 4. Bônus: testes unitários ===
def calcular_total_cliente(df):
    return df.groupBy("id_cliente").agg(_sum("valor").alias("gasto"))

# Teste simples
sample = spark.createDataFrame([(1,100),(1,200),(2,50)], ["id_cliente","valor"]) resultado = calcular_total_cliente(sample).collect()
assert resultado[0]["gasto"] in [300,50], "Erro no cálculo do total"


In [0]:
from pyspark.sql import SparkSession

# 1. Crie uma sessão Spark
spark = SparkSession.builder.appName("DataFrameExemplo").getOrCreate()

# 2. Defina os dados como uma lista de tuplas
dados = [
    ("João da Silva", "123.456.789-00", "Rua A, 123"),
    ("Maria Oliveira", "987.654.321-11", "Avenida B, 456"),
    ("Pedro Souza", "111.222.333-44", "Travessa C, 789")
]

# 3. Defina os nomes das colunas
colunas = ["nome", "cpf", "endereco"]

# 4. Crie o DataFrame
df = spark.createDataFrame(dados, colunas)

# 5. Exiba o DataFrame e o esquema
df.show()
df.printSchema()
